# 第6章 固有表現認識(Named Entity Recognition)

## 6.2 データセット・前処理・評価指標

#### 準備

In [1]:
!pip install 'datasets<4.0.0' 'transformers[ja,torch]<4.41.0'  spacy-alignments seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 78.2 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 11.4 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 44.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.7 MB/s eta 0:00:00
   ━

In [2]:
%pip uninstall -y peft

Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1


### 6.2.1 データセット

#### データセットのダウンロード

### 

In [3]:
from datasets import load_dataset

dataset = load_dataset("llm-book/ner-wikipedia-dataset")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ner-wikipedia-dataset.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [4]:
from pprint import pprint

pprint(dataset)

DatasetDict({
    train: Dataset({
        features: ['curid', 'text', 'entities'],
        num_rows: 4274
    })
    validation: Dataset({
        features: ['curid', 'text', 'entities'],
        num_rows: 534
    })
    test: Dataset({
        features: ['curid', 'text', 'entities'],
        num_rows: 535
    })
})


In [11]:
pprint(dataset["train"]["text"][0:10])

['さくら学院、Ciao Smilesのメンバー。',
 '2008年10月5日、アウェーでのレクレアティーボ・ウェルバ戦でプリメーラ・ディビシオンでの初得点を決めた。',
 'ロシア政府に近いとされ、ロシアによるSNSを用いた世論操作を行った「オリギノのトロール工場」としても知られる。',
 '宮崎市立佐土原中学校は、宮崎県宮崎市佐土原町にある公立中学校。',
 'この時点で鈴屋は日本国内に約200店舗を有していたが、これらの事業は和議申請後にも継続されるとされた。',
 '高山祭は、岐阜県高山市で毎年開催される、4月14~15日の日枝神社例祭「春の山王祭」と、10月9~10日の櫻山八幡宮例祭「秋の八幡祭」の総称である。',
 '著名人では俳優のジョニー・デップが購入して所有している。',
 '1999年2月に日立造船因島工場においてヘリコプター格納庫内に実習用講堂を設置し、その他練習艦種別変更工事を行う。',
 'そこで、陳腐化した軽戦車や、鹵獲した戦車・車輌を改造するか、または旧型戦車の生産ラインを対戦車自走砲に切り替えるという決定がなされた。',
 '優等列車であるが、一部区間では各駅に停車となる列車種別で、1938年、京阪電鉄京阪本線に設定された区間急行が最初といわれている。']


In [13]:
pprint(list(dataset["train"])[:2])

[{'curid': '3638038',
  'entities': [{'name': 'さくら学院', 'span': [0, 5], 'type': 'その他の組織名'},
               {'name': 'Ciao Smiles', 'span': [6, 17], 'type': 'その他の組織名'}],
  'text': 'さくら学院、Ciao Smilesのメンバー。'},
 {'curid': '1729527',
  'entities': [{'name': 'レクレアティーボ・ウェルバ', 'span': [17, 30], 'type': 'その他の組織名'},
               {'name': 'プリメーラ・ディビシオン', 'span': [32, 44], 'type': 'その他の組織名'}],
  'text': '2008年10月5日、アウェーでのレクレアティーボ・ウェルバ戦でプリメーラ・ディビシオンでの初得点を決めた。'}]


#### データセットの分析

In [14]:
from collections import Counter
import pandas as pd
from datasets import Dataset

def count_label_occurrences(dataset: Dataset) -> dict[str, int]:
    """固有表現タイプの出現回数をカウント"""
    # 各事例から固有表現タイプを抽出したlistを作成する
    entities = [
        e["type"] for data in dataset for e in data["entities"]
    ]

    # ラベルの出現回数が多い順に並び替える
    label_counts = dict(Counter(entities).most_common())
    return label_counts

In [15]:
label_counts_dict = {}
for split in dataset:  # 各分割セットを処理する
    label_counts_dict[split] = count_label_occurrences(dataset[split])

# DataFrame形式で表示する
df = pd.DataFrame(label_counts_dict)
df.loc["total"] = df.sum()
display(df)

,train,validation,test
人名,2394,299,287
法人名,2006,231,248
地名,1769,184,204
政治的組織名,953,121,106
製品名,934,123,158
施設名,868,103,137
その他の組織名,852,99,100
イベント名,831,85,93
total,10607,1245,1333


#### スパンの重なる固有表現の存在を判定

In [16]:
def has_overlap(spans: list[tuple[int, int]]) -> int:
    """スパンの重なる固有表現の存在を判定"""
    # スパンを開始位置で昇順に並び替える
    sorted_spans = sorted(spans, key=lambda x: x[0])
    for i in range(1, len(sorted_spans)):
        # 前のスパンの終了位置が現在の開始位置よりも大きい場合、重なっているとする
        if sorted_spans[i-1][1] > sorted_spans[i][0]:
            return 1
    return 0

In [17]:
# 各分割セットでスパンの重なる固有表現がある事例数を数える
overlap_count = 0
for split in dataset:  # 各分割セットを処理する
    for data in dataset[split]:  # 各事例数を処理する
        if data["entities"]:  # 固有表現の存在しない事例はスキップする
            # スパンのみのlistを作成する
            spans = [e["span"] for e in data["entities"]]
            overlap_count += has_overlap(spans)
    print(f"{split}におけるスパンが重複する事例数: {overlap_count}")

trainにおけるスパンが重複する事例数: 0
validationにおけるスパンが重複する事例数: 0
testにおけるスパンが重複する事例数: 0
